In [ ]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "FATAL"
import cv2
import threading
from socket import *
import time
import subprocess
from flask import Flask, Response, send_file  # 增加了 send_file 用于回传文件
import logging
from LOBOROBOT import LOBOROBOT

In [ ]:
# ================= 1. 初始化硬件 =================
print("初始化底盘与摄像头...")
bot = LOBOROBOT()
bot.t_stop(0) # 确保开机静止
# 初始化云台角度
PAN_CH, TILT_CH = 10, 9
current_pan, current_tilt = 80, 0
bot.set_servo_angle(PAN_CH, current_pan)
bot.set_servo_angle(TILT_CH, current_tilt)

In [ ]:
# ================= 2. 视频流全局机制 =================
global_frame = None
frame_lock = threading.Lock() # 视频线程锁
is_capturing = False  # 拍照锁

# 视频采集子线程
def video_capture_thread():
    global global_frame, is_capturing
    gstreamer_pipeline = (
        "libcamerasrc ! video/x-raw, width=1280, height=720, framerate=20/1 ! "
        "videoconvert ! video/x-raw, format=BGR ! appsink drop=true max-buffers=1"
    )
    cap = cv2.VideoCapture(gstreamer_pipeline, cv2.CAP_GSTREAMER)

    while True:
        if is_capturing:
            time.sleep(0.1)
            continue
        ret, frame = cap.read()
        if ret:
            frame = cv2.flip(frame, -1)
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 82]
            ret_enc, buffer = cv2.imencode('.jpg', frame, encode_param)
            if ret_enc:
                with frame_lock:
                    global_frame = buffer.tobytes()
        else:
            time.sleep(0.01)

t_cam = threading.Thread(target=video_capture_thread)
t_cam.daemon = True
t_cam.start()

In [ ]:
# ================= 3. Flask 服务器 (带拍照接口) =================
app = Flask(__name__)
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

# 接口1：实时视频流
@app.route('/mycamera')
def video_feed():
    def generate_stream():
        """视频流生成器，推送最新的 global_frame"""
        while True:
            with frame_lock:
                jpeg = global_frame
            if jpeg is not None:
                yield (b'--frame\r\n'
                       b'Content-Type: image/jpeg\r\n\r\n' + jpeg + b'\r\n')
            time.sleep(0.04)
    return Response(generate_stream(), mimetype='multipart/x-mixed-replace; boundary=frame')

# 接口2：高清晰度拍照接口 (上位机请求此 URL 即可获得高清照片)
@app.route('/capture')
def capture_photo():
    global is_capturing
    print("📸 正在执行远程拍照请求...")
    try:
        is_capturing = True
        bot.t_stop(0)  # 拍照前自动停下
        time.sleep(0.5) # 消除震动

        temp_file = "/home/pi/high_res_crack.jpg"
        # 调用底层拍照命令
        subprocess.run(["libcamera-still", "-o", temp_file, "--immediate", "--nopreview"], check=True)

        # 将拍好的高清文件直接作为 HTTP 响应发送给请求者
        return send_file(temp_file, mimetype='image/jpeg')

    except Exception as e:
        return f"Error: {str(e)}", 500
    finally:
        is_capturing = False
        print("▶️ 拍照结束，视频流恢复")

In [ ]:
# ================= 4. 网络启动 =================
def get_ip_address():
    try:
        s = socket(AF_INET, SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except:
        return "127.0.0.1"

ip = get_ip_address()
print(f"✅ 树莓派就绪！")
print(f"✅ 视频流地址: http://{ip}:8080/mycamera")
print(f"✅ 拍照下载地址: http://{ip}:8080/capture")

def run_flask():
    app.run(host='0.0.0.0', port=8080, threaded=True, use_reloader=False)

t_flask = threading.Thread(target=run_flask)
t_flask.daemon = True
t_flask.start()

In [ ]:
# ================= 5. UDP 控制循环 =================
udp_server = socket(AF_INET, SOCK_DGRAM)
udp_server.bind(('0.0.0.0', 2001))

speed = 50
try:
    while True:
        data_recv, addr = udp_server.recvfrom(1024)
        cmd = data_recv.decode('utf-8').strip()

        if cmd == "UP":          bot.t_up(speed, 0)
        elif cmd == "DOWN":      bot.t_down(speed, 0)
        elif cmd == "LEFT_MOVE": bot.moveLeft(speed, 0)
        elif cmd == "RIGHT_MOVE":bot.moveRight(speed, 0)
        elif cmd == "TURN_L":    bot.turnLeft(speed, 0)
        elif cmd == "TURN_R":    bot.turnRight(speed, 0)
        elif cmd == "STOP":      bot.t_stop(0)
        elif cmd == "CAM_UP":
            current_tilt = max(0, current_tilt - 10)
            bot.set_servo_angle(TILT_CH, current_tilt)
        elif cmd == "CAM_DOWN":
            current_tilt = min(180, current_tilt + 10)
            bot.set_servo_angle(TILT_CH, current_tilt)
        elif cmd == "CAM_LEFT":
            current_pan = min(180, current_pan + 10)
            bot.set_servo_angle(PAN_CH, current_pan)
        elif cmd == "CAM_RIGHT":
            current_pan = max(0, current_pan - 10)
            bot.set_servo_angle(PAN_CH, current_pan)

except KeyboardInterrupt:
    print("终止")
finally:
    bot.t_stop(0)
    udp_server.close()